# Day 10 · LLM 서빙 — vLLM · Ollama · NIM

관리형 API 뒤의 **서빙 계층**을 직접 다룬다. 같은 모델을 내 컴퓨터에서 띄우고, 7강 루프와 9강 앱을 base_url 교체만으로 연결한다.

> **주의 — 이 노트북은 로컬 실행이다.** Ollama가 `localhost:11434` 에 뜨기 때문에 Colab에서는 접근할 수 없다. 로컬 Jupyter 또는 VS Code에서 연다.

| 랩 | 내용 | 결과 |
|---|---|---|
| Lab 1 | Ollama 설치 → 모델 실행 → API 확인 | 내 컴퓨터에 OpenAI 호환 서버 |
| Lab 2 | 7강 에이전트 루프를 로컬 모델로 | base_url 두 줄 교체로 에이전트 동작 |
| Lab 3 (선택) | 9강 앱의 모델 스위처에 로컬 추가 | 관리형·로컬을 골라 쓰는 앱 |
| Lab 4 | LangGraph로 플래너·워커·종합 직접 구현 | 서빙 위의 코드 오케스트레이터 |

**층 구조 복습**

```
[내 앱 · 오케스트레이터]
     |  POST /v1/chat/completions
[서빙 엔진]   vLLM · Ollama · NIM
     |  배칭 · KV 캐시 관리
[GPU + 모델]
```


## Lab 1 · Ollama 로컬 서빙

### 1. 설치

```powershell
# Windows
winget install Ollama.Ollama
```
```bash
# macOS
brew install ollama
# Linux
curl -fsSL https://ollama.com/install.sh | sh
```

설치 후 **새 터미널**에서:

```bash
ollama --version
```

### 2. 모델 받기 · 실행

```bash
ollama pull qwen3:4b     # 수업 기본 — CPU(RAM 16GB)에서도 동작
ollama run qwen3:4b      # 터미널 대화 (종료: /bye)
```

GPU가 없는 것을 기본 전제로 한다 — CPU에서 qwen3:4b는 초당 10토큰 안팎으로 수업 진행이 가능하다. RAM 8GB면 `qwen3:1.7b`, GPU(VRAM 8GB+)가 있으면 `qwen3:8b`로 올린다. 에이전트 실습에는 도구 호출 지원 모델이어야 하는데, qwen3는 전 사이즈가 지원한다.

### 3. API 확인

Ollama는 `http://localhost:11434/v1` 에 **OpenAI 호환 API**를 연다. 아래 셀로 확인한다.


In [ ]:
%pip install -q openai


In [ ]:
# NVIDIA API를 부르던 코드에서 base_url·model 두 줄만 다르다
from openai import OpenAI

llm = OpenAI(
    base_url="http://localhost:11434/v1",   # ← 로컬 Ollama
    api_key="ollama",                        # 아무 문자열이면 된다
)
MODEL = "qwen3:4b"                           # 받은 모델 이름 (1.7b·8b 등)

r = llm.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "서빙 엔진이 뭔지 한 문장으로."}],
)
print(r.choices[0].message.content)


In [ ]:
# 스트리밍도 같은 규격으로 동작한다
stream = llm.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "KV 캐시를 두 문장으로 설명해줘."}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)
print()


### 관찰

- 첫 토큰까지의 시간과 생성 속도를 NVIDIA API와 비교해 본다.
- `ollama ps` 로 모델이 메모리에 올라와 있는 것을 확인한다 (GGUF 양자화 크기).

> 여기까지 되면 내 컴퓨터가 **관리형 API와 같은 규격의 서버**다. rate limit이 없다.


## Lab 2 · base_url 교체 — 7강 루프를 내 모델로

7강에서 만든 에이전트 루프(판단 → 도구 → 관찰)를 **그대로** 가져와, 모델만 로컬로 바꾼다.
아래는 도구 하나짜리 최소 루프다 — 바뀐 곳은 상단 두 줄뿐이다.


In [ ]:
import json

# ── 바뀐 곳: 이 두 줄이 전부 ──────────────────────
llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "qwen3:8b"
# ─────────────────────────────────────────────

# 도구 하나 (7강과 같은 형식)
TASKS = []
def add_task(title: str) -> dict:
    TASKS.append({"id": len(TASKS) + 1, "title": title})
    return TASKS[-1]

TOOLS = [{
    "type": "function",
    "function": {
        "name": "add_task",
        "description": "할 일을 추가한다",
        "parameters": {
            "type": "object",
            "properties": {"title": {"type": "string"}},
            "required": ["title"],
        },
    },
}]

def run_agent(question: str, max_steps: int = 5) -> str:
    messages = [
        {"role": "system", "content": "필요하면 도구로 처리한 뒤 한국어로 간결히 답하라."},
        {"role": "user", "content": question},
    ]
    for _ in range(max_steps):
        m = llm.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, temperature=0.2,
        ).choices[0].message
        if not m.tool_calls:                       # 도구를 안 부르면 종료
            return m.content
        messages.append({"role": "assistant", "content": m.content or "",
                         "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"  [로컬 모델 → 도구] {tc.function.name}({args})")
            out = add_task(**args)
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, ensure_ascii=False)})
    return "(반복 한도 초과)"

print(run_agent("'서빙 강의 복습'을 할 일로 추가해줘"))
print("TASKS:", TASKS)


### 확인할 것

- `[로컬 모델 → 도구]` 로그가 찍히면 — **tool_calls 가 네이티브로 온 것**이다. 로컬 모델이 에이전트로 동작한다.
- 도구 호출이 본문 텍스트(`<tool_call>…`)로 섞여 나오면, 그 모델이 도구 호출을 지원하지 않거나 서빙이 파싱을 안 하는 것이다 — DGX에서 겪었던 그 증상이다.

### 비교 표를 채운다

| | NVIDIA API | 로컬 Ollama |
|---|---|---|
| 첫 토큰까지 | | |
| 생성 속도 | | |
| 도구 호출 | | |
| rate limit | 있음 | 없음 |

> 같은 코드가 두 서빙에서 도는 것 자체가 오늘의 핵심이다 — OpenAI 호환 `/v1` 규격의 힘이다.


## Lab 3 · 앱에 로컬 모델 연결 (선택)

9강 앱(mcp-host)의 모델 스위처에 로컬 항목을 더한다.

```ts
// lib/models.ts
export const MODELS = [
  { id: "qwen/qwen3-next-80b-a3b-instruct",
    baseURL: "https://integrate.api.nvidia.com/v1",
    label: "NVIDIA Qwen (관리형)" },
  { id: "qwen3:4b",
    baseURL: "http://localhost:11434/v1",
    label: "로컬 Ollama" },
]
```

API Route가 선택된 `baseURL` 로 클라이언트를 만들면 끝이다. 설정 패널에서 로컬 모델을 고르고, 대화·스트리밍이 그대로 도는지 확인한다.

### 강사 시연 — DGX 도구 호출 파서

DGX의 tool_calls 텍스트 문제는 서빙 설정으로 해결된다. vLLM 기준:

```bash
vllm serve Qwen/Qwen3-... \
  --enable-auto-tool-choice \
  --tool-call-parser hermes     # Qwen 계열
```

파서를 켠 뒤 같은 에이전트 루프를 DGX로 돌리면 tool_calls 가 네이티브로 온다 — **rate limit 없는 에이전트 서버**가 된다.

### 하이브리드 라우팅 (9강의 완성)

| 역할 | 보낼 곳 | 이유 |
|---|---|---|
| 도구를 쓰는 워커 | NVIDIA API (또는 파서 켠 vLLM) | 안정적인 tool_calls |
| 요약 · 비평 · 저지 | DGX / Ollama | 도구 불필요 · 호출량 부담 없음 |
| 대량 배치 | 자가 서빙 | rate limit·비용 해방 |


## Lab 4 · LangGraph 오케스트레이터 — 플래너·워커·종합을 코드로

9강에서 하네스(Claude Code 팀)로 했던 오케스트레이터-워커 패턴을, 오늘 배운 서빙 위에서 **LangGraph 코드로** 직접 만든다.

```
START → [planner] → Send 팬아웃 → [worker]×N (병렬) → [aggregate] → END
```

- 프롬프트가 아니라 **그래프 엣지가 순서를 강제**한다 — 어길 방법이 없다.
- `max_concurrency` 설정 하나가 9강의 "동시성 캡"을 대신한다.
- 이 랩은 NVIDIA API 키만 있으면 된다 (로컬 하이브리드는 마지막에 선택).


In [ ]:
%pip install -q langgraph openai


In [ ]:
# LLM 준비 — NVIDIA Qwen (7강과 같은 방식)
import os, getpass, operator
from typing import Annotated, TypedDict
from openai import OpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...): ")
llm = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)
MODEL = "qwen/qwen3-next-80b-a3b-instruct"

def ask(prompt: str, system: str = "간결한 한국어로 답하라.") -> str:
    r = llm.chat.completions.create(
        model=MODEL, temperature=0.2, max_tokens=600,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return r.choices[0].message.content

print("LLM 준비:", MODEL)


In [ ]:
# 그래프 정의 — 상태 · 플래너 · 팬아웃 · 워커 · 종합
class State(TypedDict):
    goal: str
    tasks: list[str]
    results: Annotated[list[str], operator.add]   # 병렬 결과를 이어 붙이는 reducer
    final: str

class WorkerState(TypedDict):
    task: str

def planner(state: State):
    out = ask(f"목표를 독립적으로 처리 가능한 3개의 작은 작업으로 나눠라. 한 줄에 하나씩, 번호 없이.\n목표: {state['goal']}",
              system="계획만 세운다. 실행하지 않는다.")
    tasks = [l.strip("-· ").strip() for l in out.splitlines() if l.strip()][:3]
    print("계획:", tasks)
    return {"tasks": tasks}

def fan_out(state: State):
    # 작업 수만큼 worker 를 띄운다 — 이 리스트가 곧 병렬이다
    return [Send("worker", {"task": t}) for t in state["tasks"]]

def worker(state: WorkerState):
    print(f"  [worker 시작] {state['task'][:30]}…")
    ans = ask(f"다음 작업을 수행하라: {state['task']}")
    return {"results": [f"[{state['task']}]\n{ans}"]}

def aggregate(state: State):
    joined = "\n\n".join(state["results"])
    final = ask(f"아래 결과들을 하나의 보고로 종합하라. 중복은 정리하고 다음 단계로 마무리하라.\n\n{joined}")
    return {"final": final}

g = StateGraph(State)
g.add_node("planner", planner)
g.add_node("worker", worker)
g.add_node("aggregate", aggregate)
g.add_edge(START, "planner")
g.add_conditional_edges("planner", fan_out, ["worker"])   # 동적 팬아웃
g.add_edge("worker", "aggregate")
g.add_edge("aggregate", END)
app = g.compile()
print("그래프 컴파일 완료")


In [ ]:
# 실행 — max_concurrency 가 '동시성 캡'을 한 줄로 해결한다
out = app.invoke(
    {"goal": "AI 코딩 에이전트 도입 보고서 작성 — 현황 요약, 위험 요소, 다음 단계", "results": []},
    config={"max_concurrency": 2},   # 관리형 키의 호출 한도 대응 (9강 '호출 한도 아래의 설계')
)
print("\n===== 최종 보고 =====\n")
print(out["final"])


### 하이브리드 — 워커만 로컬 모델로 (선택)

Lab 1의 Ollama가 떠 있다면, **워커의 base_url만** 바꿔 병렬 호출을 관리형 키에서 로컬로 옮긴다. 그래프 코드는 그대로다.

```python
llm_local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL_LOCAL = "qwen3:4b"

def ask_local(prompt, system="간결한 한국어로 답하라."):
    r = llm_local.chat.completions.create(
        model=MODEL_LOCAL, temperature=0.2,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}])
    return r.choices[0].message.content

# worker 안의 ask → ask_local 로 바꾸고 그래프를 다시 compile 하면
# 병렬 워커의 호출량이 관리형 키에서 사라진다.
# 플래너·종합은 관리형(품질), 워커는 로컬(물량) — CH4 하이브리드 라우팅의 구현이다.
```

### 9강과의 비교 — 표를 채운다

| | 9강 하네스 (Claude Code 팀) | 10강 코드 (LangGraph) |
|---|---|---|
| 순서 강제 | 프롬프트 + agent teams | 그래프 엣지 (코드) |
| 병렬 | worktree · 서브에이전트 | Send 팬아웃 |
| 동시성 제한 | (수동) | `max_concurrency` |
| 모델 | Claude 구독 | 내가 고른 서빙 (교체 자유) |

> 같은 오케스트레이터-워커 패턴이다. 하네스는 개발 작업을, 이 그래프는 런타임 작업을 오케스트레이션한다.


## 산출물 체크리스트

**Lab 1 — 로컬 서빙**
- [ ] Ollama를 설치하고 `qwen3:4b`(저사양 1.7b · GPU면 8b)를 받았다
- [ ] `ollama run` 터미널 대화가 된다
- [ ] `localhost:11434/v1` 을 openai 패키지로 호출했다 — NVIDIA API와 같은 규격
- [ ] 스트리밍이 동작한다

**Lab 2 — 에이전트 연결**
- [ ] 7강 루프를 base_url 두 줄 교체로 로컬 모델에 붙였다
- [ ] tool_calls 가 네이티브로 오는 것을 확인했다
- [ ] NVIDIA API와 속도·품질을 비교해 표를 채웠다

**Lab 3 (선택) — 앱 연결**
- [ ] 앱 모델 스위처에 로컬 항목을 추가했다
- [ ] (시연) DGX에 도구 호출 파서를 켜면 tool_calls 가 살아나는 것을 봤다

**Lab 4 — LangGraph 오케스트레이터**
- [ ] planner → Send 팬아웃 → worker 병렬 → aggregate 그래프가 돌았다
- [ ] `max_concurrency=2` 로 동시성 제한을 확인했다
- [ ] (선택) 워커를 Ollama로 바꿔 하이브리드 라우팅을 확인했다
- [ ] 9강 하네스 방식과의 차이를 표로 정리했다

**개념**
- [ ] KV 캐시 · PagedAttention · continuous batching · 양자화를 각각 한 문장으로 설명할 수 있다
- [ ] Ollama / vLLM / NIM 이 각각 어떤 상황의 선택인지 안다

> 서빙까지 내려가 보면, 관리형 API의 제약이 선택의 문제였다는 것을 알게 된다.
